# CineScore V6 Master Pipeline: Relational Ingestion & Dimensionality Reduction
This pipeline mathematically squashes raw TMDB relational data into safe, generalized categorical dimensions to completely eradicate High Cardinality Overfitting.

**Core Mandates:**
- The Golden Filter: Box Office Floor set completely at $100,000.
- Global Woods: Multinational aggregation intact.
- Dimensional Reduction: Deep string extraction mapped explicitly to Big 50 Studios and Top 10 Genres.

In [8]:
import pandas as pd
import numpy as np
import os
import ast

# File Paths Logic (Google Colab vs Local)
if os.path.exists('/content'):
    print("Detected Google Colab Environment")
    DATA_DIR = "/content/Data/Raw_Dataset/"
    OUTPUT_DIR = "/content/Data/Processed_Dataset/"
else:
    print("Detected Local Environment")
    DATA_DIR = "../Data/Raw_Dataset/"
    OUTPUT_DIR = "../Data/Processed_Dataset/"

MOVIES_PATH = DATA_DIR + "movies.csv"
CAST_PATH = DATA_DIR + "cast.csv"
CREW_PATH = DATA_DIR + "crew.csv"
OUTPUT_PATH = OUTPUT_DIR + "master_analytical_df.csv"

# Load Datasets
print("Loading core datasets...")
movies = pd.read_csv(MOVIES_PATH, lineterminator='\n')
cast = pd.read_csv(CAST_PATH, lineterminator='\n')
crew = pd.read_csv(CREW_PATH, lineterminator='\n')


Detected Local Environment
Loading core datasets...


### Phase 1: The Golden Filter & Dimensionality Aggregation
We enforce the $100,000 threshold strictly at ingestion to guarantee that the Temporal Engine doesn't learn from $100 TMDB placeholder budgets.

In [9]:
print("Applying The Golden Filter & Dimensional Math...")

df = movies.copy()

# 1. Theatrical Class
df = df[(df['budget'] >= 100000) & (df['revenue'] >= 100000)]

# 2. Dimensionality Reduction: Studio Extraction (Top 50)
def extract_primary_studio(val):
    try:
        if pd.isna(val) or not isinstance(val, str):
            return 'Unknown'
        lst = ast.literal_eval(val)
        if isinstance(lst, list) and len(lst) > 0:
            return lst[0].get('name', 'Unknown') if isinstance(lst[0], dict) else str(lst[0])
        return 'Unknown'
    except Exception:
        return 'Unknown'

if 'production_companies' in df.columns:
    df['primary_studio'] = df['production_companies'].apply(extract_primary_studio)
    top_50_studios = df['primary_studio'].value_counts().nlargest(50).index
    df['primary_studio'] = df['primary_studio'].apply(lambda x: x if x in top_50_studios else 'Independent/Other')
else:
    df['primary_studio'] = 'Unknown'

# 3. Dimensionality Reduction: Genre Extraction (Top 10 Total)
df['primary_genre'] = df['genres'].apply(lambda x: str(x).split(',')[0].strip() if pd.notna(x) else 'Unknown')
top_9_genres = df['primary_genre'].value_counts().nlargest(9).index
df['primary_genre'] = df['primary_genre'].apply(lambda x: x if x in top_9_genres else 'Other')

# 4. Release Temporality (Seasonality Mapping)
df['release_date'] = pd.to_datetime(df['release_date'], errors='coerce')
df['release_month'] = df['release_date'].dt.month

def map_season(month):
    if pd.isna(month):
        return 'Dump'
    if month in [5, 6, 7]:
        return 'Summer'
    if month in [11, 12]:
        return 'Holiday'
    if month in [3, 4]:
        return 'Spring'
    return 'Dump'

df['release_season'] = df['release_month'].apply(map_season)

# 5. The Epic Window Flag
df['is_epic_window'] = ((df['runtime'] >= 130) & (df['runtime'] <= 150)).astype(int)
if 'belongs_to_collection' in df.columns:
    df['is_franchise'] = df['belongs_to_collection'].notna().astype(int)
else:
    df['is_franchise'] = 0

# Pre-computation cleanup
df['roi'] = (df['revenue'] - df['budget']) / df['budget']
df.loc[df['roi'] == np.inf, 'roi'] = np.nan

df = df.sort_values('release_date').reset_index(drop=True)


Applying The Golden Filter & Dimensional Math...


### Phase 2: Relational Flattening
Flatten relational tables into movie-level features prior to merging.

In [10]:
print("Processing Cast Data...")
top_cast = cast[cast['cast_order'] <= 2].copy()
top_cast = top_cast.sort_values(['movie_id', 'cast_order'])
cast_pivoted = top_cast.pivot_table(index='movie_id', columns='cast_order', values='person_id', aggfunc='first').reset_index()
cast_pivoted.rename(columns={0: 'actor_1_id', 1: 'actor_2_id', 2: 'actor_3_id'}, inplace=True)

print("Processing Crew Data...")
def get_first_crew(dataframe, job_titles):
    filtered = dataframe[dataframe['job'].isin(job_titles)].copy()
    filtered = filtered.sort_values(['movie_id', 'id'])
    return filtered.groupby('movie_id').first()['person_id'].reset_index()

dirs = get_first_crew(crew, ['Director']).rename(columns={'person_id': 'director_id'})
prods = get_first_crew(crew, ['Producer', 'Executive Producer']).rename(columns={'person_id': 'producer_id'})
writers = get_first_crew(crew, ['Writer', 'Screenplay']).rename(columns={'person_id': 'writer_id'})

crew_flat = dirs.merge(prods, on='movie_id', how='left').merge(writers, on='movie_id', how='left')

print("Merging relations...")
df = df.merge(cast_pivoted, left_on='id', right_on='movie_id', how='left').drop(columns=['movie_id'])
df = df.merge(crew_flat, left_on='id', right_on='movie_id', how='left').drop(columns=['movie_id'])


Processing Cast Data...
Processing Crew Data...
Merging relations...


### Phase 3: The Temporal Engine
Chronological ledgers to construct strictly pre-release historical performance.

In [11]:
print("Initializing Temporal Ledgers...")

talent_history_roi = {}
partnership_history = {}

def update_person(person_id, roi):
    if pd.isna(person_id) or pd.isna(roi):
        return
    if person_id not in talent_history_roi:
        talent_history_roi[person_id] = []
    talent_history_roi[person_id].append(roi)
    if len(talent_history_roi[person_id]) > 5:
        talent_history_roi[person_id].pop(0)

def update_pair(p1, p2, roi):
    if pd.isna(p1) or pd.isna(p2) or pd.isna(roi):
        return
    pair = tuple(sorted((p1, p2)))
    if pair not in partnership_history:
        partnership_history[pair] = []
    partnership_history[pair].append(roi)

def get_hpi(person_id):
    if pd.isna(person_id) or person_id not in talent_history_roi:
        return None
    history = talent_history_roi[person_id]
    return np.mean(history) if history else None

def get_pair_roi(p1, p2):
    if pd.isna(p1) or pd.isna(p2):
        return None
    pair = tuple(sorted((p1, p2)))
    return np.mean(partnership_history[pair]) if pair in partnership_history else None

historical_features = {
    'actor_1_hpi': [], 'actor_2_hpi': [], 'actor_3_hpi': [],
    'director_hpi': [], 'producer_hpi': [], 'writer_hpi': [],
    'cast_synergy_mult': [], 'crew_synergy_mult': [], 'lead_duo_synergy_mult': [],
    'corenswet_imputation': []
}

for idx, row in df.iterrows():
    d_hpi = get_hpi(row['director_id'])
    p_hpi = get_hpi(row['producer_id'])
    w_hpi = get_hpi(row['writer_id'])
    a1_hpi = get_hpi(row['actor_1_id'])
    a2_hpi = get_hpi(row['actor_2_id'])
    a3_hpi = get_hpi(row['actor_3_id'])

    c_12 = get_pair_roi(row['actor_1_id'], row['actor_2_id'])
    c_13 = get_pair_roi(row['actor_1_id'], row['actor_3_id'])
    c_23 = get_pair_roi(row['actor_2_id'], row['actor_3_id'])
    cast_pairs = [c for c in [c_12, c_13, c_23] if c is not None]
    avg_cast_roi = np.mean(cast_pairs) if cast_pairs else 0.0

    cr_dp = get_pair_roi(row['director_id'], row['producer_id'])
    cr_dw = get_pair_roi(row['director_id'], row['writer_id'])
    cr_pw = get_pair_roi(row['producer_id'], row['writer_id'])
    crew_pairs = [c for c in [cr_dp, cr_dw, cr_pw] if c is not None]
    avg_crew_roi = np.mean(crew_pairs) if crew_pairs else 0.0

    lead_duo = get_pair_roi(row['director_id'], row['actor_1_id'])
    avg_lead_roi = lead_duo if lead_duo is not None else 0.0

    def apply_multiplier(roi):
        return float(np.clip(1.0 + (roi * 0.1), 0.5, 2.0))

    historical_features['cast_synergy_mult'].append(apply_multiplier(avg_cast_roi) if cast_pairs else 1.0)
    historical_features['crew_synergy_mult'].append(apply_multiplier(avg_crew_roi) if crew_pairs else 1.0)
    historical_features['lead_duo_synergy_mult'].append(apply_multiplier(avg_lead_roi) if lead_duo is not None else 1.0)

    def get_count(person_id):
        return len(talent_history_roi.get(person_id, [])) if pd.notna(person_id) else 0
    a1_c = get_count(row['actor_1_id'])
    a2_c = get_count(row['actor_2_id'])
    a3_c = get_count(row['actor_3_id'])

    elite_hpi = []
    if a1_c >= 3 and a1_hpi is not None:
        elite_hpi.append(a1_hpi)
    if a2_c >= 3 and a2_hpi is not None:
        elite_hpi.append(a2_hpi)
    if a3_c >= 3 and a3_hpi is not None:
        elite_hpi.append(a3_hpi)

    mean_elite = np.mean(elite_hpi) if len(elite_hpi) > 0 else 0.0
    safe_dir = d_hpi if d_hpi is not None else 0.0
    imputed_idx = 0.8 * ((safe_dir * 0.5) + (mean_elite * 0.5))

    historical_features['actor_1_hpi'].append(imputed_idx if a1_c < 3 else a1_hpi)
    historical_features['actor_2_hpi'].append(imputed_idx if a2_c < 3 else a2_hpi)
    historical_features['actor_3_hpi'].append(imputed_idx if a3_c < 3 else a3_hpi)
    historical_features['director_hpi'].append(d_hpi)
    historical_features['producer_hpi'].append(p_hpi)
    historical_features['writer_hpi'].append(w_hpi)
    historical_features['corenswet_imputation'].append(imputed_idx)

    # Commit updates to ledger
    current_roi = row['roi']
    if not pd.isna(current_roi):
        update_person(row['director_id'], current_roi)
        update_person(row['producer_id'], current_roi)
        update_person(row['writer_id'], current_roi)
        update_person(row['actor_1_id'], current_roi)
        update_person(row['actor_2_id'], current_roi)
        update_person(row['actor_3_id'], current_roi)

        update_pair(row['actor_1_id'], row['actor_2_id'], current_roi)
        update_pair(row['actor_1_id'], row['actor_3_id'], current_roi)
        update_pair(row['actor_2_id'], row['actor_3_id'], current_roi)

        update_pair(row['director_id'], row['producer_id'], current_roi)
        update_pair(row['director_id'], row['writer_id'], current_roi)
        update_pair(row['producer_id'], row['writer_id'], current_roi)
        update_pair(row['director_id'], row['actor_1_id'], current_roi)

for col, values in historical_features.items():
    df[col] = list(values)

# V6 Synergy & HPI Imputation Constraint
hpi_cols = ['actor_1_hpi', 'actor_2_hpi', 'actor_3_hpi', 'director_hpi', 'producer_hpi', 'writer_hpi', 'corenswet_imputation']
mult_cols = ['cast_synergy_mult', 'crew_synergy_mult', 'lead_duo_synergy_mult']
for col in hpi_cols:
    df[col] = df[col].fillna(0.0)
    df[col] = np.clip(df[col], -1.0, 10.0)
for col in mult_cols:
    df[col] = df[col].fillna(1.0)


Initializing Temporal Ledgers...


In [12]:
print("Pipeline Processing Complete. Dataset shape:", df.shape)

os.makedirs(OUTPUT_DIR, exist_ok=True)
df.to_csv(OUTPUT_PATH, index=False)
print(f"Data successfully exported to {OUTPUT_PATH}")


Pipeline Processing Complete. Dataset shape: (3351, 45)
Data successfully exported to ../Data/Processed_Dataset/master_analytical_df.csv
